# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
1. 1, 4, 1.33... один пропуск сумма известных чисел равна 4 после замены пропуска на 0 среднее равно 4/3

In [ ]:
2. 2 после удаления пробелов и перевода в нижний регистр остаются два разных значения a и b

In [ ]:
3. 3 drop_duplicates() создал новую таблицу но ее не сохранили в df поэтому длина исходной таблицы не изменилась

In [ ]:
4. 1005 пробел в 1 000 удаляется строки превращаются в числа 1000+5

In [ ]:
5. [9, 10] первая дата относится к сентябрю вторая к октябрю

In [1]:
import pandas as pd

In [2]:
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())

1 1.3333333333333333


In [3]:
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())

2


In [4]:
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))

3


In [5]:
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())

1005


In [7]:
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

[9, 10]


In [8]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [15]:
import pandas as pd

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print("после чтения:", len(df))

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("после очистки сумм:", len(df))

    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("после очистки городов:", len(df))

    df = df.drop_duplicates().copy()
    print("после удаления дублей:", len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print("после обработки дат:", len(df))

    df = df.loc[df["customer_id"] != "TEST"].copy()
    print("после удаления TEST:", len(df))

    print("пропусков в amount:", df["amount"].isna().sum())
    print("пропусков в city:", df["city"].isna().sum())
    print("order_id уникален:", df["order_id"].is_unique)

    return df

clean_df = clean("orders.csv")

После чтения: 5463
После очистки сумм: 5463
После очистки городов: 5463
После удаления дублей: 5283
После обработки дат: 5283
После удаления TEST: 5280
Пропусков в amount: 0
Пропусков в city: 0
order_id уникален: True


останется 5280 строк, отменённые заказы сохранены они нужны для дальнейшего анализа отмен. 
Проверки должны показать 0 пропусков в amount и city а для order_id True

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [ ]:
# 2

In [18]:
sep_paid = clean_df.loc[
    (clean_df["month"] == 9) & (clean_df["status"] == "оплачен")
]
oct_paid = clean_df.loc[
    (clean_df["month"] == 10) & (clean_df["status"] == "оплачен")
]

sep = sep_paid["amount"].sum()
oct = oct_paid["amount"].sum()

sep, oct, round((oct / sep - 1) * 100, 1)

(np.float64(35243410.0), np.float64(34419520.0), np.float64(-2.3))

In [19]:
cities = ["Алматы", "Астана", "Шымкент", "Караганда", "Актобе", "Атырау"]

sep_city = pd.Series([
    sep_paid.loc[sep_paid["city"] == "Алматы", "amount"].sum(),
    sep_paid.loc[sep_paid["city"] == "Астана", "amount"].sum(),
    sep_paid.loc[sep_paid["city"] == "Шымкент", "amount"].sum(),
    sep_paid.loc[sep_paid["city"] == "Караганда", "amount"].sum(),
    sep_paid.loc[sep_paid["city"] == "Актобе", "amount"].sum(),
    sep_paid.loc[sep_paid["city"] == "Атырау", "amount"].sum()
], index=cities)

oct_city = pd.Series([
    oct_paid.loc[oct_paid["city"] == "Алматы", "amount"].sum(),
    oct_paid.loc[oct_paid["city"] == "Астана", "amount"].sum(),
    oct_paid.loc[oct_paid["city"] == "Шымкент", "amount"].sum(),
    oct_paid.loc[oct_paid["city"] == "Караганда", "amount"].sum(),
    oct_paid.loc[oct_paid["city"] == "Актобе", "amount"].sum(),
    oct_paid.loc[oct_paid["city"] == "Атырау", "amount"].sum()
], index=cities)

result = pd.DataFrame({"Сентябрь": sep_city, "Октябрь": oct_city})
result["Изменение, %"] = ((result["Октябрь"] / result["Сентябрь"] - 1) * 100).round(1)
result

,Сентябрь,Октябрь,"Изменение, %"
Алматы,14501070.0,14643020.0,1.0
Астана,9034220.0,8850530.0,-2.0
Шымкент,4558650.0,4044010.0,-11.3
Караганда,3648520.0,3371680.0,-7.6
Актобе,2576960.0,2540420.0,-1.4
Атырау,923990.0,969860.0,5.0


общая выручка снизилась на 2.3% сильнее всего  в шымкенте на 11.3%. Таблица по городам должна совпасть с расчётом урока 4

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [ ]:
# 3
                  Шаг                                      Убрано строк     Осталось строк     Изменение выручки октябрь/сентябрь
Исходные данные: только оплаченные для расчёта выручки          0              5463                      −22,7%
Перевели суммы в числа, привели города и даты к единому виду    0              5463                      −22,7%
Удалили дубли                                                  180             5283                      −19,6%
Удалили заказы TEST                                             3              5280                      −2,3%             

сильнее всего цифру исказили 3 заказа TEST: после их удаления рассчитанное падение изменилось с −19.6% до −2.3%
Число строк здесь включает все статусы выручка рассчитана только по оплаченным заказам

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [20]:
missing = clean_df["delivery_days"].isna()

In [21]:
missing.sum(), clean_df.loc[missing, "city"].value_counts(), clean_df.loc[missing, "status"].value_counts()

(np.int64(158),
 city
 Алматы       63
 Астана       40
 Караганда    24
 Шымкент      22
 Атырау        5
 Актобе        4
 Name: count, dtype: int64,
 status
 оплачен    144
 отменён     14
 Name: count, dtype: int64)

In [22]:
d = clean_df.loc[clean_df["city"] == "Шымкент", "delivery_days"]

In [23]:
round((d.dropna() <= 2).mean() * 100, 2), round((d.fillna(d.median()) <= 2).mean() * 100, 2), round((d.fillna(0) <= 2).mean() * 100, 2)

(np.float64(29.84), np.float64(28.99), np.float64(31.83))

честнее указать 29.84% среди заказов с известным сроком доставки и пояснить что у 22 заказов шымкента срок неизвестен. 
Медиана подставляет предполагаемый срок а ноль ошибочно считает неизвестную доставку мгновенной

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [24]:
df = pd.read_csv("orders.csv")

In [25]:
df = df.drop_duplicates().copy()

In [26]:
df["city"] = df["city"].str.strip().str.lower().map(CITY)

In [27]:
df["city"].nunique(), len(df)

(6, 5283)

Получится 6 городов и 5283 строки CITY это словарь из задания 1
Если для итоговой очистки также убрать клиента TEST останется 5280 строк в показанном коде аи этого шага не было

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [28]:
days = clean_df["order_date"].dt.day_name().value_counts()

In [29]:
days

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64

In [30]:
round((786 - 684) / 786 * 100, 1)

13.0

в понедельник заказов на 13.0% меньше чем в самый загруженный четверг
при составлении графика курьеров на четверг стоит предусмотреть больше сотрудников чем на понедельник

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [31]:
sh = clean_df.loc[clean_df["city"] == "Шымкент"].sort_values("order_date")

In [32]:
sep_mean = sh.loc[sh["month"] == 9, "delivery_days"].mean()

In [33]:
oct_mean = sh.loc[sh["month"] == 10, "delivery_days"].mean()

In [34]:
round(sep_mean, 2), round(oct_mean, 2)

(np.float64(2.03), np.float64(6.54))

In [35]:
last_fast = sh.loc[sh["delivery_days"] < 5, "order_date"].max()

In [36]:
first_slow = sh.loc[sh["order_date"] > last_fast, "order_date"].min()

In [37]:
last_fast, first_slow

(Timestamp('2026-09-30 22:00:00'), Timestamp('2026-10-01 10:00:00'))

In [38]:
other = clean_df.loc[clean_df["city"] != "Шымкент"]

In [39]:
sep30 = other.loc[
    (other["order_date"] >= "2026-09-30") &
    (other["order_date"] < "2026-10-01"),
    "delivery_days"
].mean()

In [40]:
oct1 = other.loc[
    (other["order_date"] >= "2026-10-01") &
    (other["order_date"] < "2026-10-02"),
    "delivery_days"
].mean()

In [41]:
round(sep30, 2), round(oct1, 2)

(np.float64(1.83), np.float64(1.89))

В остальных городах средний срок составил 1.83 дня 30 сентября и 1.89 дня 1 октября особого скачка нет
с 1 октября проблема проявляется именно в шымкенте
в первую очередь нужно проверить что изменилось в местной доставке на рубеже сентября и октября

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [42]:
def to_amount(s):
    s = s.str.replace(" ", "").str.replace(",", ".")
    return pd.to_numeric(s, errors="coerce")

In [43]:
example = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])

In [44]:
to_amount(example)

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64

In [45]:
raw = pd.read_csv("orders.csv")

In [46]:
amount = to_amount(raw["amount"])

In [47]:
amount.isna().sum(), amount.sum()

(np.int64(0), np.float64(109883360.0))

 0 пропусков и 109883360 все суммы исходного файла удалось перевести в числа без потери выручки